# Cuaderno de validación

**Tarea python para desarrolladores** (Máster Big Data & Data Engineering, UCM)

Este cuaderno comprueba, con datos reales del [portal de datos abiertos del Ayuntamiento de Madrid](https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/downloads), que las clases del paquete funcionan como pide el enunciado y que lanzan las excepciones correspondientes (`CacheError` y `MadridError`).

**Índice**

1. Importación del paquete
2. Módulo `cache`: clase `Cache`
3. Módulo `cache`: clase `CacheURL`
4. Módulo `madridFines`: función `get_url`
5. Módulo `madridFines`: clase `MadridFines`

## 1. Importación del paquete

In [1]:
import time
from pathlib import Path

import tarea_python_ucm
import tarea_python_ucm.cache as cache
import tarea_python_ucm.madridFines as madridFines
from tarea_python_ucm import Cache, CacheURL, CacheError, MadridFines, MadridError, get_url

In [2]:
print("Class names:", tarea_python_ucm.__all__)
print("Cache folder (CACHE_DIR):", cache.CACHE_DIR)
print("Downloads page (DOWNLOAD):", madridFines.DOWNLOAD)

Class names: ['Cache', 'CacheURL', 'CacheError', 'MadridFines', 'MadridError', 'get_url']
Cache folder (CACHE_DIR): C:\Users\aleja\.my_cache
Downloads page (DOWNLOAD): https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/downloads


Todo el paquete está documentado con docstrings en formato Google. Por ejemplo, la ayuda de `get_url` incluye sus parámetros, el valor devuelto, las excepciones y dos ejemplos que se ejecutan como doctest:

In [3]:
help(get_url)

Help on function get_url in module tarea_python_ucm.madridFines:

get_url(year: int, month: int) -> str
    Retrieves the URL from Datos Madrid based on given year and month.

    Args:
        year (int): Year you are looking for.
        month (int): Month you are looking for.

    Returns:
        str: Full URL for download.

    Raises:
        MadridError: If month not between 1 and 12.
        MadridError: If the date is before June 2017.
        MadridError: If the downloads page cannot be reached or its status is not 200.
        MadridError: If the month is not on the page or has no download link.

    Examples:
        >>> get_url(2024, 13)
        Traceback (most recent call last):
            ...
        tarea_python_ucm.madridFines.MadridError: Month must be between 1 and 12.
        >>> get_url(2017, 5)
        Traceback (most recent call last):
            ...
        tarea_python_ucm.madridFines.MadridError: Date must be from June 2017.



## 2. Módulo `cache`: clase `Cache`

In [4]:
c = Cache("validacion", 1)
c.clear()

print(c)          # __str__
print(repr(c))    # __repr__
print("app_name:", c.app_name)
print("obsolescence:", c.obsolescence)
print("cache_dir:", c.cache_dir)

Cache-validacion,
Obsolescence of 1 days.
Path in C:\Users\aleja\.my_cache\validacion.
Cache('validacion', 1)
app_name: validacion
obsolescence: 1
cache_dir: C:\Users\aleja\.my_cache\validacion


### 2.1 Guardar, comprobar y leer

`set` guarda el dato con un nombre (si ya existe, lo sobrescribe), `exists` comprueba si está y `load` lo recupera.

In [5]:
c.set("greeting", "hi, Madrid")
print("exists('greeting'):", c.exists("greeting"))
print("exists('other'):", c.exists("other"))
print("load('greeting'):", c.load("greeting"))

c.set("greeting", "adiós, Madrid")
print("load after overwriting:", c.load("greeting"))

exists('greeting'): True
exists('other'): False
load('greeting'): hi, Madrid
load after overwriting: adiós, Madrid


### 2.2 Edad y obsolescencia

- **`how_old`** devuelve la edad del fichero en milisegundos.
- **`is_obsolete`** la compara con `obsolescence` (1 día en este objeto), pasado a milisegundos.

Recién guardado, el fichero no está obsoleto. El caso contrario se comprueba en los tests (`tests/test_cache.py`), que simulan el paso de dos días sin tener que esperar.

In [6]:
print(f"how_old: {c.how_old('greeting')} ms")
print("is_obsolete:", c.is_obsolete("greeting"))

how_old: 8.203125 ms
is_obsolete: False


### 2.3 Borrar

`delete` borra un fichero; si no existe, no lanza ninguna excepción. `clear` borra todos los ficheros de la aplicación.

In [7]:
c.delete("greeting")
print("exists after delete:", c.exists("greeting"))
c.delete("greeting")
print("delete of a missing name: no error")

c.set("a", "1")
c.set("b", "2")
print("files before clear:", c.exists("a"), c.exists("b"))
c.clear()
print("files after clear:", c.exists("a"), c.exists("b"))

exists after delete: False
delete of a missing name: no error
files before clear: True True
files after clear: False False


### 2.4 Excepciones `CacheError`

Se lanza `CacheError` en estos casos:

- el constructor recibe un `app_name` vacío o que no es texto, o una `obsolescence` que no es un entero mayor que 0;
- se lee o se consulta la edad de un nombre que no está guardado;
- se intenta guardar un dato vacío o con un nombre vacío.

In [8]:
invalid = [("", 1), (123, 1), ("validacion", 0), ("validacion", -3), ("validacion", True), ("validacion", "7")]
for app_name, obsolescence in invalid:
    try:
        Cache(app_name, obsolescence)
    except CacheError as e:
        print(f"CacheError: {e}")

CacheError: app_name cannot be empty or null!
CacheError: app_name cannot be empty or null!
CacheError: obsolescence must be greater than 0!
CacheError: obsolescence must be greater than 0!
CacheError: obsolescence must be greater than 0!
CacheError: obsolescence must be greater than 0!


In [9]:
try:
    c.load("missing")
except CacheError as e:
    print("load('missing') -> CacheError:", e)

try:
    c.how_old("missing")
except CacheError as e:
    print("how_old('missing') -> CacheError:", e)

try:
    c.set("empty", "")
except CacheError as e:
    print("set with empty data -> CacheError:", e)

try:
    c.set("", "hi")
except CacheError as e:
    print("set with empty name -> CacheError:", e)

load('missing') -> CacheError: 'missing' does not exist in cache!
how_old('missing') -> CacheError: 'missing' does not exist in cache!
set with empty data -> CacheError: data cannot be empty or null!
set with empty name -> CacheError: name cannot be empty or null!


## 3. Módulo `cache`: clase `CacheURL`

`CacheURL` hereda de `Cache` y trabaja con URLs:

- **Nombre del fichero.** Es el hash MD5 de la URL junto con sus parámetros opcionales (`**kwargs`), un nombre corto y válido en cualquier sistema operativo.
- **`get(url)`.** Si la URL está guardada y no es obsoleta, la lee del disco; si no, la descarga, la guarda y la devuelve.

In [10]:
cu = CacheURL("validacion_url", 1)
cu.clear()
url = madridFines.DOWNLOAD

print(repr(cu))
print("hash of the url:", cu._hash_me(url))

CacheURL('validacion_url', 1)
hash of the url: d12976ef826db5ee7fa52d134c40a69a


In [11]:
start = time.time()
html = cu.get(url)
first = time.time() - start

start = time.time()
html_again = cu.get(url)
second = time.time() - start

print(f"1st get (download): {first:.2f} s")
print(f"2nd get (cache): {second:.4f} s")

1st get (download): 10.02 s
2nd get (cache): 0.0023 s


Los métodos redefinidos (`exists`, `load`, `how_old`, `is_obsolete`, `delete`) reciben la URL y, opcionalmente, los parámetros de la petición. La misma URL con otros parámetros es otro fichero.

In [12]:
print("exists(url):", cu.exists(url))
print("exists(url, timeout=5):", cu.exists(url, timeout=5))
print("load(url) is the same text:", cu.load(url) == html)
print(f"how_old(url): {cu.how_old(url):.0f} ms")
print("is_obsolete(url):", cu.is_obsolete(url))

cu.delete(url)
print("exists after delete:", cu.exists(url))

exists(url): True
exists(url, timeout=5): False
load(url) is the same text: True
how_old(url): 22 ms
is_obsolete(url): False
exists after delete: False


### 3.1 Excepciones `CacheError` en `CacheURL`

Se lanza `CacheError` en estos casos:

- la web responde con un código distinto de 200 (aquí un 404);
- no se puede conectar con el servidor;
- se lee una URL que no está guardada;
- la URL está vacía.

Cuando falla la descarga no se guarda nada.

In [13]:
try:
    cu.get("https://datos.madrid.es/dataset/no-existe-este-dataset")
except CacheError as e:
    print("404 -> CacheError:", e)

try:
    cu.get("https://servidor-inexistente.invalid")
except CacheError as e:
    print("No connection -> CacheError:", str(e)[:120], "...")

try:
    cu.load(url)
except CacheError as e:
    print("load of a url not in the cache -> CacheError:", e)

try:
    cu.get("")
except CacheError as e:
    print("empty url -> CacheError:", e)

print("files in the cache:", [p.name for p in Path(cu.cache_dir).iterdir()])

404 -> CacheError: 'https://datos.madrid.es/dataset/no-existe-este-dataset' returned status 404
No connection -> CacheError: Unable to reach 'https://servidor-inexistente.invalid': HTTPSConnectionPool(host='servidor-inexistente.invalid', port=44 ...
load of a url not in the cache -> CacheError: 'https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/downloads' does not exist in cache!
empty url -> CacheError: url cannot be empty or null!
files in the cache: []


## 4. Módulo `madridFines`: función `get_url`

`get_url(year, month)` hace *scraping* de la página de descargas con `requests` y BeautifulSoup:

1. Busca el enlace «CSV Detalle. {Mes} {año}».
2. Después busca el botón «Descarga» de ese mismo recurso.
3. Devuelve `RAIZ` + la ruta de descarga.

En mayo de 2019 la página tiene el fichero en TXT y en CSV, y la función devuelve el CSV.

In [14]:
url_dec_2024 = get_url(2024, 12)
url_may_2019 = get_url(2019, 5)
print(url_dec_2024)
print(url_may_2019)
print("Both are CSV download links:", url_dec_2024.endswith(".csv") and url_may_2019.endswith(".csv"))

https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv.csv
https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-91-multas-circulacion-detalle-csv/download/210104-91-multas-circulacion-detalle-csv.csv
Both are CSV download links: True


### 4.1 Excepciones `MadridError` en `get_url`

Se lanza `MadridError` en estos casos:

- el mes no está entre 1 y 12;
- la fecha es anterior a junio de 2017 (estos dos se comprueban antes de conectarse a la web);
- el mes todavía no está publicado en la página.

In [15]:
for year, month in [(2024, 0), (2024, 13), (2017, 5), (2016, 12), (2030, 1)]:
    try:
        get_url(year, month)
    except MadridError as e:
        print(f"get_url({year}, {month}) -> MadridError: {e}")

get_url(2024, 0) -> MadridError: Month must be between 1 and 12.
get_url(2024, 13) -> MadridError: Month must be between 1 and 12.
get_url(2017, 5) -> MadridError: Date must be from June 2017.
get_url(2016, 12) -> MadridError: Date must be from June 2017.
get_url(2030, 1) -> MadridError: 'CSV Detalle. Enero 2030' not found.


## 5. Módulo `madridFines`: clase `MadridFines`

### 5.1 Constructor y atributos

El constructor recibe `app_name` y `obsolescence`, que se usan para crear el `CacheURL`.

- Al empezar, `data` es un DataFrame vacío y `loaded` una lista vacía.
- Los tres atributos son privados y se consultan con propiedades.
- Si la configuración de la caché no es válida, el `CacheError` se relanza como `MadridError`.

In [16]:
fines = MadridFines("multas", 7)
print("cacheurl:", repr(fines.cacheurl))
print("data is empty:", fines.data.empty)
print("loaded:", fines.loaded)

cacheurl: CacheURL('multas', 7)
data is empty: True
loaded: []


In [17]:
for app_name, obsolescence in [("", 7), ("multas", 0)]:
    try:
        MadridFines(app_name, obsolescence)
    except MadridError as e:
        print(f"MadridError: {e}")

MadridError: Invalid cache configuration: app_name cannot be empty or null!
MadridError: Invalid cache configuration: obsolescence must be greater than 0!


Sin datos cargados, las tres consultas lanzan `MadridError`:

In [18]:
try:
    fines.fines_calification()
except MadridError as e:
    print("fines_calification() -> MadridError:", e)

try:
    fines.total_payment()
except MadridError as e:
    print("total_payment() -> MadridError:", e)

try:
    fines.fines_hour("no_data.png")
except MadridError as e:
    print("fines_hour() -> MadridError:", e)

fines_calification() -> MadridError: No data loaded: use add first.
total_payment() -> MadridError: No data loaded: use add first.
fines_hour() -> MadridError: No data loaded: use add first.


### 5.2 Métodos estáticos `load` y `clean`

`load` usa el `CacheURL` que recibe para obtener el CSV de un mes y lo devuelve **sin limpiar**: los nombres de columna todavía tienen espacios (`' PUNTOS'`, `'VEL_CIRCULA '`…).

In [19]:
raw = MadridFines.load(2024, 12, fines.cacheurl)
print("shape:", raw.shape)
print(list(raw.columns))
raw.head(3)

shape: (249801, 14)
['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO', ' PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA ', 'COORDENADA-X', 'COORDENADA-Y                                                                                                                                          ']


,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA. ...,,,,
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",,,,
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACI�N DE UN...,,,,


`clean` aplica la limpieza de la ETAPA 1 sobre el mismo DataFrame y no devuelve nada (`None`):

In [20]:
result = MadridFines.clean(raw)
print("clean returns:", result)
print(list(raw.columns))
print("CALIFICACION values:", sorted(raw["CALIFICACION"].unique()))
print(raw[["VEL_LIMITE", "VEL_CIRCULA", "COORDENADA-X", "COORDENADA-Y"]].dtypes)
print("index:", raw.index.name, "| sorted:", raw.index.is_monotonic_increasing)
raw.head(3)

clean returns: None
['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO', 'PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA', 'COORDENADA-X', 'COORDENADA-Y']
CALIFICACION values: ['GRAVE', 'LEVE', 'MUY GRAVE']
VEL_LIMITE      float64
VEL_CIRCULA     float64
COORDENADA-X    float64
COORDENADA-Y    float64
dtype: object
index: FECHA | sorted: True


,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
FECHA,,,,,,,,,,,,,,
2024-12-01,GRAVE,AV DE ANDALUCIA 19 C1,12,2024,0.0,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,441073.42,4468726.53
2024-12-01,GRAVE,C FUENTE CARRANTONA 44 C2,12,2024,0.0,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,446846.27,4474115.01
2024-12-01,GRAVE,AV DE ANDALUCIA 50 C2,12,2024,0.0,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,441090.00,4468177.59


### 5.3 Añadir meses: `add`

`add(year, month)` hace varias cosas:

- descarga el mes con `load` (o lo lee de la caché);
- lo limpia con `clean`;
- lo une a `data` y lo ordena por fecha;
- registra `(month, year)` en `loaded`.

Las columnas `COORDENADA_X` / `COORDENADA_Y`, se unifican como `COORDENADA-X` / `COORDENADA-Y`.

In [21]:
fines.add(2024, 12)
fines.add(2019, 5)

data = fines.data
print("loaded:", fines.loaded)
print("rows:", len(data), "| December 2024:", (data["ANIO"] == 2024).sum(), "| May 2019:", (data["ANIO"] == 2019).sum())
print("from", data.index.min(), "to", data.index.max())

loaded: [(12, 2024), (5, 2019)]
rows: 422689 | December 2024: 249801 | May 2019: 172888
from 2019-05-01 00:00:00 to 2024-12-01 23:59:00


Si se añade un mes que ya está cargado, no se hace nada.

`data` y `loaded` devuelven una **copia**, así que modificar lo que devuelven no cambia el objeto.

In [22]:
rows_before = len(fines.data)
fines.add(2024, 12)
print("loaded:", fines.loaded, "| same number of rows:", len(fines.data) == rows_before)

months = fines.loaded
months.append((1, 2020))
print("modified copy:", months)
print("loaded in the object:", fines.loaded)

loaded: [(12, 2024), (5, 2019)] | same number of rows: True
modified copy: [(12, 2024), (5, 2019), (1, 2020)]
loaded in the object: [(12, 2024), (5, 2019)]


Si se pide un mes concreto que no es válido o no está disponible, `add` lanza `MadridError` y no cambia nada:

In [23]:
for year, month in [(2024, 13), (2016, 1), (2030, 1)]:
    try:
        fines.add(year, month)
    except MadridError as e:
        print(f"add({year}, {month}) -> MadridError: {e}")
print("loaded:", fines.loaded)

add(2024, 13) -> MadridError: Month must be between 1 and 12.
add(2016, 1) -> MadridError: Date must be from June 2017.
add(2030, 1) -> MadridError: 'CSV Detalle. Enero 2030' not found.
loaded: [(12, 2024), (5, 2019)]


### 5.4 Año completo: `add(year)`

Con `month=None` se añaden todos los meses publicados del año. Los meses que no están disponibles (anteriores a junio de 2017 o aún no publicados) se omiten sin error.

In [24]:
fines_2026 = MadridFines("multas", 7)
fines_2026.add(2026)
print("loaded:", fines_2026.loaded)
print("rows:", len(fines_2026.data))

loaded: [(1, 2026), (2, 2026), (3, 2026), (4, 2026)]
rows: 785023


### 5.5 Consulta `fines_calification`

Número de multas de cada tipo (`CALIFICACION`) para cada mes y año cargado:

In [25]:
table = fines.fines_calification()
table

,CALIFICACION,GRAVE,LEVE,MUY GRAVE
MES,ANIO,,,
5,2019,61124,111017,747
12,2024,157605,91388,808


### 5.6 Consulta `total_payment`

Importe total por mes y año:

- **MAX**: todos los infractores pagan el importe completo.
- **MIN**: las multas con `DESCUENTO == "SI"` se pagan con el 50 % de descuento por pronto pago.

In [26]:
fines.total_payment()

,,MAX,MIN
MES,ANIO,,
5,2019,17207760.0,8603880.0
12,2024,37005230.0,18502615.0


In [27]:
fines_2026.total_payment()

,,MAX,MIN
MES,ANIO,,
1,2026,25445740.0,12722870.0
2,2026,25469440.0,12734720.0
3,2026,27719930.0,13859965.0
4,2026,22441850.0,11220925.0


### 5.7 Gráfico `fines_hour`

Número de multas por hora del día, con una línea por cada mes cargado. El gráfico se guarda con el nombre indicado (en la carpeta de este cuaderno) y aquí se muestra la imagen guardada.

In [28]:
fines.fines_hour("evolucion_multas.png")

![](evolucion_multas.png)

In [29]:
fines_2026.fines_hour("evolucion_multas_2026.png")

![](evolucion_multas_2026.png)

### 5.8 Borrar la caché: `clean_cache`

`clean_cache` borra todos los ficheros descargados de la aplicación. No modifica `data` ni `loaded`.

In [30]:
demo = MadridFines("validacion_madrid", 7)
demo.cacheurl.get(madridFines.DOWNLOAD)
demo.clean_cache()

Ficheros guardados en la caché de `multas`: cada uno es un CSV descargado y su nombre es el hash de la URL.

In [31]:
for p in sorted(Path(fines.cacheurl.cache_dir).iterdir()):
    print(f"{p.name}  {p.stat().st_size / 1000000:.1f} MB")

383f9136f675316987af131a87552757  1.1 MB
3adafe4411183833f67d461a71bd9176  49.6 MB
5b90a5d24237477ead46f8a36f680e4a  54.3 MB
5c5a6dbabc06466acd33d89325a5bf48  61.1 MB
80e38d076cb6fe1b5b3e5145f44af08a  71.5 MB
c57ee18c6e4e1778cfa52b4b85a693e2  45.7 MB
f8b50023e09580ccd3cf1150fee3c167  57.0 MB
